# 9jaLingo vLLM TTS — GPU Test (Colab)

This notebook tests the 9jaLingo vLLM TTS server on a Colab GPU.

**Requirements:** Select **GPU runtime** (Runtime → Change runtime type → T4 GPU)

## Setup

In [ ]:
# Check GPU is available
!nvidia-smi

In [ ]:
# Clone the repo (replace with your fork if private)
!git clone https://github.com/9jaLingo/9jalingo-vllm.git
%cd 9jalingo-vllm

In [ ]:
# Set HuggingFace token for private model access
# Option 1: Use Colab secrets (recommended)
try:
    from google.colab import userdata
    import os
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN loaded from Colab secrets')
except Exception:
    print('⚠️  Could not load from Colab secrets.')
    print('Set it manually below:')

# Option 2: Set manually (uncomment and paste your token)
# import os
# os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'

In [ ]:
# Install dependencies (GPU mode)
# 1. Install the project + its deps (naijalingo-tts-2, fastapi, etc.)
!pip install -e . --quiet

# 2. Install vLLM (GPU version)
!pip install vllm --quiet

print('\n✅ All dependencies installed')

In [ ]:
# Verify GPU is visible to PyTorch and vLLM
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

import vllm
print(f'vLLM: {vllm.__version__}')

## Prepare Model

Download the HF model and strip custom weights incompatible with vLLM's `Lfm2ForCausalLM`.

In [ ]:
from prepare_model import prepare

model_path = prepare()
print(f'\n✅ vLLM-compatible model ready at: {model_path}')

## Start the Server

Launch the FastAPI server in the background, then test it with HTTP requests.

In [ ]:
# Start the server in background
import subprocess, time, os

env = os.environ.copy()
server_proc = subprocess.Popen(
    ['python', 'server.py'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for server to be ready
print('Starting server... (this takes ~1-2 min for model loading + warmup)')
import urllib.request
for i in range(180):  # wait up to 3 min
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:8000/health')
        print(f'\n✅ Server is ready! (took ~{(i+1)*2}s)')
        break
    except Exception:
        print('.', end='', flush=True)
else:
    print('\n❌ Server did not start in time. Check logs below:')
    server_proc.terminate()
    out, _ = server_proc.communicate(timeout=5)
    print(out[-3000:] if out else 'No output')

In [ ]:
# Quick health check + list available speakers
import requests

# Health
r = requests.get('http://localhost:8000/health')
print('Health:', r.json())

# Speakers
r = requests.get('http://localhost:8000/v1/speakers')
speakers = r.json()
print(f"\nSpeakers loaded: {speakers['total']}")
print(f"By language: {speakers['by_language']}")
print(f"\nSample speakers: {speakers['speakers'][:5]}")

## Test 1: Basic TTS (Language Tag Only)

Uses the vLLM fast path — no speaker embedding, just language tag routing.

In [ ]:
import requests, soundfile as sf, io, os
from IPython.display import Audio, display

OUTPUT_DIR = "tts_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def tts_request(text, voice='pcm', speaker=None, save_as=None, **kwargs):
    """Make a TTS request, return audio data + sample rate, optionally save to file."""
    payload = {
        'text': text,
        'model': '9javox',
        'voice': voice,
        'response_format': 'wav',
        'temperature': kwargs.get('temperature', 1.0),
        'top_p': kwargs.get('top_p', 0.95),
        'repetition_penalty': kwargs.get('repetition_penalty', 1.1),
    }
    if speaker:
        payload['speaker'] = speaker
    
    resp = requests.post('http://localhost:8000/v1/audio/speech', json=payload)
    resp.raise_for_status()
    data, sr = sf.read(io.BytesIO(resp.content))
    
    if save_as:
        out_path = os.path.join(OUTPUT_DIR, save_as)
        sf.write(out_path, data, sr)
        print(f'Saved: {out_path}')
    
    return data, sr

# Nigerian Pidgin
text_pcm = "My people, if patience were food, Nigerians would never go hungry!"
data, sr = tts_request(text_pcm, voice='pcm', save_as='pidgin_test.wav')
print(f'Pidgin: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# Yoruba
text_yo = "Ẹ kú ilé o, a dúpẹ́ fún ọjọ́ tuntun yìí."
data, sr = tts_request(text_yo, voice='yo', save_as='yoruba_test.wav')
print(f'Yoruba: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# Hausa
text_ha = "Sannu da zuwa, muna farin ciki da ganin ku."
data, sr = tts_request(text_ha, voice='ha', save_as='hausa_test.wav')
print(f'Hausa: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# Igbo
text_ig = "Nnọọ, anyị nwere obi ụtọ ịhụ unu."
data, sr = tts_request(text_ig, voice='ig', save_as='igbo_test.wav')
print(f'Igbo: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

## Test 2: Speaker Embedding Path

Uses the Direct model path with a pre-computed speaker embedding.
This loads a second model copy — requires ~3GB+ VRAM on top of the vLLM engine.

In [ ]:
# Test with a specific speaker (uses Direct model path)
text = "Wetin dey happen, how body na?"
data, sr = tts_request(text, voice='pcm', speaker='ada_pcm', save_as='speaker_ada_pcm.wav')
print(f'Speaker ada_pcm: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# Another speaker
text = "Ẹ̀yin ọmọ Yorùbá, ẹ kú iṣẹ́ o!"
data, sr = tts_request(text, voice='yo', speaker='adeola_yo', save_as='speaker_adeola_yo.wav')
print(f'Speaker adeola_yo: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

## Test 3: Long-form Generation

Tests sentence chunking + silence insertion for longer text.

In [ ]:
long_text = (
    "My people, if patience were food, Nigerians would never go hungry! "
    "But since we cannot cook patience for dinner, let us cook progress instead "
    "and season it with hard work, unity, and small small enjoyment. "
    "As we dey go, make we no forget say the journey wey start with one step, "
    "fit carry us to where we never imagine before."
)
print(f'Text length: {len(long_text)} chars, estimated: {len(long_text)/15:.1f}s')

data, sr = tts_request(long_text, voice='pcm', save_as='long_form_test.wav')
print(f'Long-form (vLLM): {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# Test long-form with speaker embedding (Direct model path)
data, sr = tts_request(long_text, voice='pcm', speaker='ada_pcm', save_as='long_form_speaker_test.wav')
print(f'Long-form (Direct/speaker): {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

## Test 4: Performance Benchmark (RTF)

Measure Real-Time Factor: how fast the GPU generates compared to audio duration.

In [ ]:
import time

texts = {
    'pcm': "How far, everything dey alright?",
    'yo': "Bawo ni, ṣe daadaa ni?",
    'ha': "Ina kwana, lafiya lau?",
    'ig': "Kedụ, ọ dị mma?",
}

print(f'{"Lang":<6} {"Gen Time":<12} {"Audio Dur":<12} {"RTF":<8}')
print('-' * 40)

for lang, text in texts.items():
    start = time.time()
    data, sr = tts_request(text, voice=lang)
    gen_time = time.time() - start
    audio_dur = len(data) / sr
    rtf = gen_time / audio_dur if audio_dur > 0 else 0
    print(f'{lang:<6} {gen_time:<12.2f} {audio_dur:<12.2f} {rtf:<8.2f}')

print('\nRTF < 1.0 = faster than real-time ✅')

## Cleanup

In [ ]:
# Stop the server
server_proc.terminate()
server_proc.wait(timeout=10)
print('✅ Server stopped')